BƯỚC 1: CẤU HÌNH ĐƯỜNG DẪN & TẠO THƯ MỤC

In [ ]:
import os
from google.colab import drive

# 1. Kết nối Drive
try:
    drive.mount('/content/drive')
except:
    pass

# 2. Định nghĩa các đường dẫn
BASE_DIR = "/content/drive/MyDrive/DoAn_NIDS/Dataset/"

# Đường dẫn lấy dữ liệu
DATA_PATH = os.path.join(BASE_DIR, "Binary_Data/")

# Đường dẫn lưu Model
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "Models-DNN/")

# 3. Tạo thư mục nếu chưa có
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

print("-" * 50)
print("✅ CẤU HÌNH THÀNH CÔNG!")
print(f"   📂 Nơi lấy dữ liệu: {DATA_PATH}")
print(f"   📂 Nơi lưu Model  : {MODEL_SAVE_PATH}")
print("-" * 50)

Bước 2: Nạp dữ liệu

In [ ]:
import joblib
import os

print("-" * 50)
print("⏳ BƯỚC 2: ĐANG NẠP DỮ LIỆU TỪ KHO BINARY...")

# 1. Load dữ liệu Train
X_train = joblib.load(DATA_PATH + 'X_train.pkl')
y_train = joblib.load(DATA_PATH + 'y_train.pkl')

# 2. Load Trọng số
class_weights = joblib.load(DATA_PATH + 'class_weights.pkl')

# 3. Kiểm tra thông số
print("\n✅ ĐÃ NẠP XONG! KIỂM TRA KÍCH THƯỚC:")
print(f"   - X_train shape: {X_train.shape}")
print(f"     -> Số lượng mẫu: {X_train.shape[0]}")
print(f"     -> Số đặc trưng (Cột): {X_train.shape[1]}")
print(f"   - y_train shape: {y_train.shape}")

print("\n⚖️ TRỌNG SỐ ÁP DỤNG:")
print(class_weights)
print("-" * 50)

BƯỚC 3: XÂY DỰNG KIẾN TRÚC DNN

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

print("-" * 50)
print("🏗️ BƯỚC 3: ĐANG DỰNG KHUNG MÔ HÌNH DNN...")

def build_dnn_model(input_dim):
    model = Sequential(name="DNN_Binary_Standard")

    # --- LỚP ẨN 1: MỞ RỘNG (512) ---
    model.add(Dense(512, input_dim=input_dim, activation='relu'))
    model.add(BatchNormalization()) # Giúp mô hình hội tụ nhanh hơn
    model.add(Dropout(0.2))         # Tắt 20% nơ-ron để ép mô hình học kỹ hơn

    # --- LỚP ẨN 2: CO DẦN (256) ---
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))         # Tăng dropout lên 30% ở các lớp sâu

    # --- LỚP ẨN 3: CÔ ĐỌNG (128) ---
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))

    # --- LỚP OUTPUT: QUYẾT ĐỊNH (1) ---
    # Sigmoid trả về xác suất từ 0 đến 1
    # < 0.5: Sạch (0)
    # >= 0.5: Tấn công (1)
    model.add(Dense(1, activation='sigmoid'))

    # --- COMPILE (LẮP RÁP) ---
    # Adam: Thuật toán tối ưu tốt nhất hiện nay cho dữ liệu bảng
    # lr=0.001: Tốc độ học tiêu chuẩn
    optimizer = Adam(learning_rate=0.001)

    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy', # Hàm mất mát chuẩn cho 2 lớp
                  metrics=['accuracy'])
    return model

# Khởi tạo mô hình
# X_train.shape[1] chính là số cột (37)
model = build_dnn_model(X_train.shape[1])

# In bảng tóm tắt kiến trúc
model.summary()

print("-" * 50)
print("✅ MÔ HÌNH ĐÃ SẴN SÀNG ĐỂ TRAIN!")

BƯỚC 4: THIẾT LẬP "BỘ BA GIÁM SÁT" (CALLBACKS)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import os

print("-" * 50)
print("👮 BƯỚC 4: THIẾT LẬP CÁC GIÁM SÁT VIÊN (CALLBACKS)...")

# Định nghĩa đường dẫn lưu file model tốt nhất
checkpoint_path = os.path.join(MODEL_SAVE_PATH, 'Scenario1_DNN_Best.keras')

callbacks = [
    # 1. ModelCheckpoint: CHỈ LƯU HỌC SINH GIỎI NHẤT
    # Nó sẽ canh 'val_loss'. Khi nào thấy loss thấp kỷ lục thì mới lưu đè file.
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True, # Quan trọng: Không lưu model lởm
        mode='min',          # Loss càng nhỏ càng tốt
        verbose=1
    ),

    # 2. EarlyStopping: DỪNG KHI KHÔNG TIẾN BỘ
    # Nếu sau 10 vòng (patience=10) mà val_loss không giảm -> Dừng luôn
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True, # Quay về trạng thái tốt nhất trước khi dừng
        verbose=1
    ),

    # 3. ReduceLROnPlateau: GIẢM TỐC KHI GẶP KHÓ
    # Nếu sau 3 vòng (patience=3) mà loss đi ngang -> Giảm tốc độ học đi một nửa (factor=0.5)
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=0.00001, # Không giảm quá sâu dưới mức này
        verbose=1
    )
]

print("✅ Đã tuyển dụng xong 3 giám sát viên:")
print(f"   1. Checkpoint (Lưu tại: {checkpoint_path})")
print("   2. EarlyStopping (Kiên nhẫn 10 vòng)")
print("   3. ReduceLR (Giảm tốc sau 3 vòng tắc nghẽn)")
print("-" * 50)

BƯỚC 5: HUẤN LUYỆN MÔ HÌNH

In [ ]:
import matplotlib.pyplot as plt

print("-" * 50)
print("🚀 BƯỚC 5: BẮT ĐẦU HUẤN LUYỆN (TRAINING)...")
print("   - Đang chạy với Batch Size: 1024")
print("   - Đang sử dụng Class Weights để cân bằng")
print("-" * 50)

# 1. Thực hiện lệnh Train
history = model.fit(
    X_train, y_train,
    validation_split=0.1, # Cắt 10% tập train làm tập đánh giá (Validate)
    epochs=50,            # Số vòng tối đa (thường sẽ dừng sớm hơn nhờ EarlyStopping)
    batch_size=256,      # Đọc 256 mẫu một lúc
    class_weight=class_weights, # Áp dụng trọng số
    callbacks=callbacks,  # Gọi bộ 3 giám sát viên vào làm việc
    verbose=1             # Hiện thanh tiến trình
)

print("\n✅ HUẤN LUYỆN HOÀN TẤT!")

# 2. Vẽ biểu đồ đánh giá (Loss & Accuracy)
plt.figure(figsize=(14, 6))

# --- Biểu đồ Loss (Hàm mất mát) ---
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss', color='blue', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss', color='orange', linewidth=2)
plt.title('Hàm mất mát (Loss) - Càng thấp càng tốt')
plt.xlabel('Vòng (Epochs)')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# --- Biểu đồ Accuracy (Độ chính xác) ---
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc', color='green', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Val Acc', color='red', linewidth=2)
plt.title('Độ chính xác (Accuracy) - Càng cao càng tốt')
plt.xlabel('Vòng (Epochs)')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.show()

BƯỚC 6: KIỂM TRA MÔ HÌNH TRÊN TẬP TEST

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import joblib

print("-" * 50)
print("🧐 BƯỚC 6: KIỂM TRA MÔ HÌNH TRÊN TẬP TEST (Dữ liệu chưa từng gặp)...")

# 1. Load dữ liệu Test
# Lưu ý: Đây là tập X_test_scaled gốc và y_test đã mã hóa 0/1
X_test = joblib.load(DATA_PATH + 'X_test.pkl')
y_test = joblib.load(DATA_PATH + 'y_test.pkl')

print(f"   - Số lượng mẫu Test: {len(y_test)}")

# 2. Dự đoán
# Model trả về xác suất (VD: 0.95, 0.02...), ta cần làm tròn
# > 0.5 -> 1 (Tấn công), <= 0.5 -> 0 (Sạch)
print("   - Đang thực hiện dự đoán...")
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype("int32")

# 3. Đánh giá tổng quan
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\n🎯 KẾT QUẢ CUỐI CÙNG:")
print(f"   - Test Accuracy: {accuracy * 100:.2f}%")
print(f"   - Test Loss: {loss:.4f}")

# 4. Báo cáo chi tiết (Precision, Recall, F1-Score)
print("\n📊 BẢNG BÁO CÁO CHI TIẾT:")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Attack (1)']))

# 5. Ma trận nhầm lẫn (Confusion Matrix)
print("heatmap Ma trận nhầm lẫn:")
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Benign', 'Attack'],
            yticklabels=['Benign', 'Attack'])
plt.xlabel('Dự đoán (Predicted)')
plt.ylabel('Thực tế (Actual)')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# 4. CHẠY ĐÁNH GIÁ (Phần code mới bổ sung)
if 'model' in locals():
    print("-" * 50)
    print("🚀 BẮT ĐẦU ĐÁNH GIÁ LẠI...")

    # Dự đoán
    start_time = time.time()
    y_pred_prob = model.predict(X_test)
    end_time = time.time()

    # Chuyển xác suất thành nhãn (0 hoặc 1)
    y_pred = (y_pred_prob > 0.5).astype("int32")

    # --- A. Báo cáo chỉ số ---
    print("\n📊 CLASSIFICATION REPORT:")
    print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack']))

    # --- B. Tính các chỉ số nâng cao ---
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Tỷ lệ báo động giả (FPR)
    fpr_score = fp / (fp + tn)
    print(f"⚠️ Tỷ lệ báo động giả (FPR): {fpr_score * 100:.2f}%")

    # Tốc độ xử lý
    time_per_sample = (end_time - start_time) / len(y_test)
    print(f"⏱️ Tốc độ xử lý: {time_per_sample * 1000000:.2f} µs/mẫu")

    # --- C. Vẽ biểu đồ ROC ---
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
import time
import numpy as np

def calculate_inference_time(model, X_data, num_samples=1000):
    # Lấy ngẫu nhiên num_samples mẫu để test
    indices = np.random.choice(len(X_data), num_samples, replace=False)
    X_sample = X_data[indices]

    # Warm-up (chạy nháp vài lần để GPU/CPU nóng máy)
    _ = model.predict(X_sample[:10], verbose=0)

    # Bắt đầu đo
    start_time = time.time()
    _ = model.predict(X_sample, verbose=0)
    end_time = time.time()

    # Tính toán
    total_time = end_time - start_time
    time_per_sample = (total_time / num_samples) * 1_000_000 # Đổi sang microseconds (µs)

    return time_per_sample

print("-" * 50)
print("⏱️ ĐANG ĐO TỐC ĐỘ XỬ LÝ (INFERENCE TIME)...")

# 1. Đo DNN
try:
    dnn_time = calculate_inference_time(model, X_test, 1000)
    print(f"   - DNN Speed: {dnn_time:.2f} µs/gói tin")
except NameError:
    print("   - Không tìm thấy biến model trong RAM (Cần load lại nếu đã tắt session)")
    dnn_time = 0 # Giá trị giả định để điền bảng

# 2. Đo 1D-CNN (Hiện tại chỉ có mô hình DNN, nên sẽ dùng lại model cho mục đích đo lường)
try:
    cnn_time = calculate_inference_time(model, X_test, 1000)
    print(f"   - 1D-CNN Speed: {cnn_time:.2f} µs/gói tin")
except NameError:
    print("   - Không tìm thấy biến model (CNN) trong RAM")
    cnn_time = 0

print("-" * 50)
